# Analisi e Tracking dello Spostamento Provini (Video Delta)

Questo notebook implementa la selezione interattiva della ROI e il tracciamento 1D verticale (con meccanismo anti-occlusione) dei provini di allungamento sotto carico.
I video si trovano nella cartella `dati/delta` e sono denominati `MVI_40XX.MP4`.

In linea con le specifiche di `agent.md`, l'intero notebook lavora esclusivamente con gli **ID dei provini** (es. `101`, `301`, `991`) e mappa automaticamente l'ID al rispettivo file video.

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
from tqdm.auto import tqdm
import json

# Configurazione percorsi
DELTA_DIR = Path("dati/delta")
TRACK_DIR = Path("dati/tracking")
CONFIG_ROI_FILE = TRACK_DIR / "roi_allungamento_config.json"

# Creazione automatica delle cartelle se non esistono
TRACK_DIR.mkdir(parents=True, exist_ok=True)
print("✅ Percorsi configurati e cartelle di tracking verificate.")

✅ Percorsi configurati e cartelle di tracking verificate.


In [2]:
# Tabella di associazione: Codice Provino -> Nome File Video in dati/delta
MAP_PROVINO_TO_VIDEO = {
    "991": "MVI_4064.MP4",
    "101": "MVI_4065.MP4",
    "601": "MVI_4066.MP4",
    "301": "MVI_4067.MP4",
    "901": "MVI_4068.MP4",
    "992": "MVI_4070.MP4",
    "602": "MVI_4071.MP4",
    "302": "MVI_4072.MP4",
    "102": "MVI_4073.MP4",
    "902": "MVI_4074.MP4",
    "303": "MVI_4075.MP4",
    "304": "MVI_4076.MP4",
}

# Rilevamento dei video effettivamente presenti su disco
disponibili = {}
for provino_id, video_name in MAP_PROVINO_TO_VIDEO.items():
    video_path = DELTA_DIR / video_name
    if video_path.exists():
        disponibili[provino_id] = video_path

print(f"Provini disponibili rilevati in dati/delta ({len(disponibili)}): {list(disponibili.keys())}")
if len(disponibili) < len(MAP_PROVINO_TO_VIDEO):
    non_trovati = set(MAP_PROVINO_TO_VIDEO.keys()) - set(disponibili.keys())
    print(f"⚠️ Nota: {len(non_trovati)} video non sono presenti in delta (es. {list(non_trovati)})")

Provini disponibili rilevati in dati/delta (0): []
⚠️ Nota: 12 video non sono presenti in delta (es. ['304', '601', '101', '302', '303', '902', '991', '102', '301', '901', '992', '602'])


## 1. Parametri di Scala e Calibrazione
La costante `pixel_to_mm` serve a convertire lo spostamento in pixel in allungamento fisico in millimetri. Il valore standard calibrato è `0.0491` mm/pixel.

In [3]:
# Valore di default per la conversione
pixel_to_mm = 0.0491
print(f"Scala di conversione configurata: {pixel_to_mm} mm/pixel")

Scala di conversione configurata: 0.0491 mm/pixel


*(Opzionale)* Se desideri ricalibrare manualmente la scala, imposta `calibrazione_manuale = True` nella cella sottostante per aprire una finestra OpenCV in cui tracciare una linea di riferimento di `44.65 mm` (es. la larghezza nota del provino).

In [4]:
calibrazione_manuale = False
lunghezza_nota_mm = 44.65

if calibrazione_manuale and disponibili:
    # Usa il primo provino disponibile come riferimento per la calibrazione
    primo_provino = list(disponibili.keys())[0]
    vid_path = disponibili[primo_provino]
    
    cap = cv2.VideoCapture(str(vid_path))
    ret, frame_init = cap.read()
    cap.release()

    if ret:
        punti = []
        img_base = frame_init.copy()

        def seleziona_punti(event, x, y, flags, param):
            img_display = img_base.copy()
            h, w = img_display.shape[:2]

            if event == cv2.EVENT_LBUTTONDOWN:
                punti.append((x, y))
                cv2.circle(img_base, (x, y), 8, (0, 0, 255), -1)
                if len(punti) == 2:
                    cv2.line(img_base, punti[0], punti[1], (0, 255, 0), 4)
                img_display = img_base.copy()

            # Disegna croce mirino
            cv2.line(img_display, (0, y), (w, y), (0, 255, 255), 2)
            cv2.line(img_display, (x, 0), (x, h), (0, 255, 255), 2)
            cv2.imshow("Calibrazione: Traccia linea", img_display)

        cv2.namedWindow("Calibrazione: Traccia linea", cv2.WINDOW_NORMAL)
        cv2.resizeWindow("Calibrazione: Traccia linea", 1280, 720)
        cv2.imshow("Calibrazione: Traccia linea", img_base)
        cv2.setMouseCallback("Calibrazione: Traccia linea", seleziona_punti)
        
        print("👆 Clicca su 2 punti dell'immagine per definire la scala in mm, poi premi un tasto qualsiasi per confermare.")
        cv2.waitKey(0)
        cv2.destroyAllWindows()

        if len(punti) >= 2:
            distanza_px = np.hypot(punti[1][0] - punti[0][0], punti[1][1] - punti[0][1])
            pixel_to_mm = lunghezza_nota_mm / distanza_px
            print(f"✅ Scala calcolata: {pixel_to_mm:.5f} mm/pixel")
        else:
            print("⚠️ Punti insufficienti. Utilizzo del default.")

## 2. Selezione Interattiva della ROI (Region of Interest)
La ROI selezionata servirà come template da tracciare durante il video. Le coordinate della ROI vengono salvate in un file di configurazione JSON (`dati/tracking/roi_allungamento_config.json`) in modo da non dover ripetere la selezione ad ogni riavvio.

In [5]:
# Carica o inizializza la configurazione delle ROI
config_roi = {}
if CONFIG_ROI_FILE.exists():
    try:
        with open(CONFIG_ROI_FILE, 'r') as f:
            config_roi = json.load(f)
        print(f"✅ Configurazione ROI caricata. Provini configurati: {list(config_roi.keys())}")
    except Exception as e:
        print(f"⚠️ Impossibile leggere il file di configurazione: {e}. Verrà inizializzato nuovo.")

# Avvia la selezione per i provini che non hanno ancora una ROI definita
forza_riselezione = False  # Cambia a True se vuoi rifare la selezione di tutti

for provino_id, video_path in disponibili.items():
    if provino_id in config_roi and not forza_riselezione:
        continue
        
    cap = cv2.VideoCapture(str(video_path))
    ret, first_frame = cap.read()
    cap.release()
    
    if not ret:
        print(f"❌ Impossibile leggere il primo frame del provino {provino_id}.")
        continue
        
    nome_finestra = f"SELEZIONE ROI - Provino: {provino_id}"
    cv2.namedWindow(nome_finestra, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(nome_finestra, 1280, 720)
    
    print(f"👉 Seleziona l'area da tracciare per il provino '{provino_id}' e premi INVIO. (ESC per saltare)")
    bbox = cv2.selectROI(nome_finestra, first_frame, False)
    cv2.destroyWindow(nome_finestra)
    
    # Ciclo di waitKey per garantire la chiusura della finestra su Windows
    for _ in range(5):
        cv2.waitKey(1)
        
    if bbox != (0, 0, 0, 0):
        config_roi[provino_id] = {
            'bbox': [int(v) for v in bbox]
        }
        print(f"✅ ROI configurata per provino {provino_id}: {bbox}")
    else:
        print(f"⚠️ Selezione ROI saltata per provino {provino_id}.")

# Salva la configurazione finale
with open(CONFIG_ROI_FILE, 'w') as f:
    json.dump(config_roi, f, indent=4)
print("✅ File di configurazione ROI aggiornato.")

✅ File di configurazione ROI aggiornato.


## 3. Algoritmo di Tracking 1D Verticale con Anti-Occlusione
Il tracking si sposta solo lungo l'asse Y (la coordinata X rimane fissa). Se l'area di ricerca presenta un match template con punteggio inferiore a `0.65` (ad esempio per l'intrusione di una mano), il tracking entra in stato di occlusione e mantiene l'ultima posizione valida.

In [6]:
dati_tracking = {}

for provino_id, video_path in disponibili.items():
    save_path = TRACK_DIR / f"tracking_full_{provino_id}.npy"
    
    # Carica da cache se già elaborato
    if save_path.exists():
        dati_tracking[provino_id] = np.load(save_path, allow_pickle=True).item()
        print(f"✅ Risultati di tracking caricati da cache per provino {provino_id}.")
        continue
        
    if provino_id not in config_roi:
        print(f"⚠️ ROI non configurata per il provino {provino_id}. Salto.")
        continue
        
    bbox = config_roi[provino_id]['bbox']
    
    cap = cv2.VideoCapture(str(video_path))
    ret, first_frame = cap.read()
    if not ret:
        print(f"❌ Impossibile leggere il primo frame per {provino_id}.")
        cap.release()
        continue
        
    gray_first = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
    x_fisso, y, w, h = bbox
    template = gray_first[y:y+h, x_fisso:x_fisso+w]
    y_iniziale_centro = int(y + h / 2)
    ultimo_y = y
    
    MARGINE_RICERCA_Y = 100
    SOGLIA_CONFIDENZA = 0.65
    
    spostamenti = []
    maschera_occlusione = []
    
    tot_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    
    print(f"🚀 Avvio tracking provino {provino_id} ({tot_frames} frame)...")
    
    with tqdm(total=tot_frames, desc=f"Analisi Provino {provino_id}", leave=True) as pbar:
        while True:
            ret, frame = cap.read()
            if not ret: 
                break
            
            gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            H_frame = gray_frame.shape[0]
            
            # Finestra di ricerca verticale bloccata in X
            sx, ex = x_fisso, x_fisso + w
            sy = max(0, ultimo_y - MARGINE_RICERCA_Y)
            ey = min(H_frame, ultimo_y + h + MARGINE_RICERCA_Y)
            
            search_area = gray_frame[sy:ey, sx:ex]
            
            occluso = True
            if search_area.shape[0] >= h and search_area.shape[1] == w:
                res = cv2.matchTemplate(search_area, template, cv2.TM_CCOEFF_NORMED)
                _, max_val, _, max_loc = cv2.minMaxLoc(res)
                
                if max_val >= SOGLIA_CONFIDENZA:
                    ultimo_y = sy + max_loc[1]
                    occluso = False
            
            y_att = ultimo_y + h / 2
            valore_mm = (y_att - y_iniziale_centro) * pixel_to_mm
            
            spostamenti.append(valore_mm)
            maschera_occlusione.append(occluso)
            pbar.update(1)
            
    cap.release()
    
    pacchetto_dati = {
        'spostamento': np.array(spostamenti),
        'occluso': np.array(maschera_occlusione),
        'pixel_to_mm': pixel_to_mm,
        'tipo_tracking': '1D_Vertical_AntiOcclusion',
        'bbox': bbox
    }
    
    dati_tracking[provino_id] = pacchetto_dati
    np.save(save_path, pacchetto_dati)
    print(f"💾 Salvato tracking provino {provino_id} in {save_path}")
print("\n🎉 Tutti i provini disponibili sono stati elaborati!")


🎉 Tutti i provini disponibili sono stati elaborati!


## 4. Visualizzazione Grafica dello Spostamento
Visualizza l'andamento dello spostamento del provino nel tempo. I tratti blu indicano il tracciamento attivo, mentre i tratti arancioni indicano i momenti in cui l'oggetto è stato coperto (occluso) e la coordinata è stata tenuta in memoria.

In [7]:
def mostra_tracking_bicolore(provino_id):
    if provino_id not in dati_tracking:
        print(f"⚠️ Dati non disponibili per il provino {provino_id}")
        return
        
    pacchetto = dati_tracking[provino_id]
    y = pacchetto['spostamento']
    mask = pacchetto['occluso']
    x = np.arange(len(y))

    clear_output(wait=True)
    with plt.ioff():
        fig = plt.figure(figsize=(12, 5))
        
        # 1. Linea principale blu (tracking attivo)
        plt.plot(x, y, color='#1f77b4', linewidth=2, label='Tracking Attivo', zorder=1)
        
        # 2. Sovrapposizione arancione (occlusione/memoria)
        y_occluso = np.where(mask, y, np.nan)
        plt.plot(x, y_occluso, color='#ff7f0e', linewidth=3, label='Occlusione (Memoria)', zorder=2)

        plt.title(f"Spostamento Provino nel Tempo - ID: {provino_id}", fontsize=13, fontweight='bold')
        plt.xlabel("Frame", fontsize=11)
        plt.ylabel("Spostamento (mm)", fontsize=11)
        plt.grid(True, alpha=0.3, linestyle='--')
        plt.legend(fontsize=10)
        display(fig)
        plt.close(fig)

provini_ids = sorted(list(dati_tracking.keys()))
if provini_ids:
    widgets.interact(mostra_tracking_bicolore, provino_id=widgets.Dropdown(options=provini_ids, value=provini_ids[0], description="Provino:"))
else:
    print("Nessun provino elaborato.")


Nessun provino elaborato.


## 5. Anteprima Video con Overlay Grafico in Tempo Reale
Mostra il video in una finestra separata sovrapponendo gli elementi del tracking:
- **Box verde/giallo**: La ROI attualmente tracciata (gialla se occlusa).
- **Tunnel fucsia**: La finestra di ricerca verticale (`ultimo_y` +/- 100 pixel).
- **Linea e freccia rossa**: Il vettore dello spostamento calcolato rispetto alla posizione iniziale.

In [8]:
def avvia_anteprima(provino_id):
    if provino_id not in disponibili:
        print(f"❌ Video per il provino {provino_id} non trovato.")
        return
    if provino_id not in dati_tracking:
        print(f"⚠️ Esegui prima il tracking per il provino {provino_id}.")
        return
        
    video_path = disponibili[provino_id]
    pacchetto = dati_tracking[provino_id]
    spostamento = pacchetto['spostamento']
    occluso = pacchetto['occluso']
    bbox = pacchetto['bbox']
    
    cap = cv2.VideoCapture(str(video_path))
    ret, first_frame = cap.read()
    if not ret:
        print("❌ Errore nella lettura del video.")
        cap.release()
        return
        
    nome_finestra = f"Anteprima Tracking 1D - Provino: {provino_id}"
    cv2.namedWindow(nome_finestra, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(nome_finestra, 1280, 720)
    
    x_fisso, y, w, h = bbox
    x_iniziale_centro = int(x_fisso + w / 2)
    y_iniziale_centro = int(y + h / 2)
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    frame_idx = 0
    
    print("📺 Anteprima video in corso... Premi 'q' all'interno della finestra del video per uscire.")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: 
            break
            
        H_frame = frame.shape[0]
        
        if frame_idx < len(spostamento):
            defl_mm = spostamento[frame_idx]
            is_occluded = occluso[frame_idx]
            
            # Calcola l'attuale Y
            y_att = int(y_iniziale_centro + (defl_mm / pixel_to_mm))
            ultimo_y = y_att - h // 2
            
            # Tunnel verticale fucsia
            MARGINE_RICERCA_Y = 100
            sy = max(0, ultimo_y - MARGINE_RICERCA_Y)
            ey = min(H_frame, ultimo_y + h + MARGINE_RICERCA_Y)
            
            # 1. Riferimento orizzontale zero (Blu)
            cv2.line(frame, (x_iniziale_centro - 60, y_iniziale_centro), (x_iniziale_centro + 60, y_iniziale_centro), (255, 0, 0), 2)
            # 2. Linea verticale binario (Blu)
            cv2.line(frame, (x_iniziale_centro, 0), (x_iniziale_centro, H_frame), (255, 0, 0), 1)
            # 3. Finestra di ricerca verticale (Fucsia)
            cv2.rectangle(frame, (x_fisso, sy), (x_fisso + w, ey), (255, 0, 255), 1)
            
            # 4. Freccia rossa deflessione
            if abs(y_att - y_iniziale_centro) > 2:
                cv2.arrowedLine(frame, (x_iniziale_centro, y_iniziale_centro), (x_iniziale_centro, y_att), (0, 0, 255), 3, tipLength=0.2)
                
            # 5. Rettangolo ROI e testo stato
            p1 = (x_fisso, ultimo_y)
            p2 = (x_fisso + w, ultimo_y + h)
            if not is_occluded:
                cv2.rectangle(frame, p1, p2, (0, 255, 0), 2)
                cv2.putText(frame, f"Spostamento: {defl_mm:.2f} mm", (50, 80), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
            else:
                cv2.rectangle(frame, p1, p2, (0, 255, 255), 2)
                cv2.putText(frame, f"OCCLUSIONE: {defl_mm:.2f} mm (Memoria)", (50, 80), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)
                            
        cv2.imshow(nome_finestra, frame)
        frame_idx += 1
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    cap.release()
    cv2.destroyAllWindows()
    for _ in range(5):
        cv2.waitKey(1)
    print("✅ Riproduzione terminata.")

if provini_ids:
    widgets.interact_manual(avvia_anteprima, provino_id=widgets.Dropdown(options=provini_ids, value=provini_ids[0], description="Riproduci:"))
else:
    print("Nessun provino tracciato.")

Nessun provino tracciato.


## 6. Verifica dei Carichi con agent.md

In questa sezione implementiamo un motore di verifica e classificazione per controllare se i dati delle tabelle presenti in `agent.md` (in particolare i carichi di rottura teorici e la composizione dei pesi applicati) coincidono con i dati misurati dal sistema di tracking dell'allungamento.

L'algoritmo rileva i plateau di stabilità della curva di spostamento, stima la costante di compliance specifica del provino ($C$ in mm/kg) tramite regressione/grid search sui livelli noti, ed effettua la mappatura automatica dei plateau rilevati nei carichi discreti standard (`0.0, 0.5, 1.25, 1.75, 2.0, 2.5, 3.25, 3.75` kg).

In [9]:
# Dati teorici estratti da agent.md
BREAKING_LOADS = {
    "991": 3.25,
    "992": 2.50,
    "101": 2.50,
    "102": 2.50,
    "301": 1.75,
    "302": 2.00,
    "303": 1.75,
    "304": 2.00,
    "601": 2.00,
    "602": 2.00,
    "901": 1.75,
    "902": 1.75,
}

WEIGHT_COMPOSITIONS = {
    0.0: "Nessun carico",
    0.5: "0.5 kg",
    1.25: "1.25 kg",
    1.75: "1.25 kg + 0.5 kg",
    2.0: "2 kg",
    2.5: "2 kg + 0.5 kg",
    3.25: "2 kg + 1.25 kg",
    3.75: "2 kg + 1.25 kg + 0.5 kg",
}

def get_expected_loads(breaking_load):
    targets = [0.5, 1.25, 1.75, 2.0, 2.5, 3.25, 3.75]
    return [t for t in targets if t <= breaking_load]

def get_expected_seq(expected_loads):
    seq = [0.0]
    for load in expected_loads:
        if load == 0.5:
            seq.extend([0.5, 0.0])
        elif load == 1.25:
            seq.extend([1.25, 0.0])
        elif load == 1.75:
            seq.extend([0.5, 1.75, 0.0])
        elif load == 2.0:
            seq.extend([2.0, 0.0])
        elif load == 2.5:
            seq.extend([0.5, 2.5, 0.0])
        elif load == 3.25:
            seq.extend([1.25, 3.25, 0.0])
        elif load == 3.75:
            seq.extend([0.5, 1.75, 3.75, 0.0])
    if seq[-1] == 0.0:
        seq.pop()
    return seq

def is_subsequence(sub, main):
    it = iter(main)
    return all(x in it for x in sub)

def detect_plateaus_verif(y, occluded, window_size=50, std_threshold=0.08, min_plateau_len=50):
    n = len(y)
    stable = np.zeros(n, dtype=bool)
    for i in range(n - window_size):
        if not np.any(occluded[i:i+window_size]) and np.std(y[i:i+window_size]) < std_threshold:
            stable[i:i+window_size] = True
            
    plateaus = []
    in_plateau = False
    start_idx = 0
    for i in range(n):
        if stable[i] and not in_plateau:
            in_plateau = True
            start_idx = i
        elif not stable[i] and in_plateau:
            in_plateau = False
            length = i - start_idx
            if length >= min_plateau_len:
                plateaus.append((start_idx, i, length, np.mean(y[start_idx:i])))
    if in_plateau:
        length = n - start_idx
        if length >= min_plateau_len:
            plateaus.append((start_idx, n, length, np.mean(y[start_idx:])))
            
    filtered = []
    for plat in plateaus:
        if not filtered:
            filtered.append(plat)
        else:
            prev = filtered[-1]
            if abs(plat[3] - prev[3]) < 0.25:
                new_start = prev[0]
                new_end = plat[1]
                new_len = new_end - new_start
                new_mean = (prev[3]*prev[2] + plat[3]*plat[2]) / (prev[2] + plat[2])
                filtered[-1] = (new_start, new_end, new_len, new_mean)
            else:
                filtered.append(plat)
    return filtered

def analyze_specimen_verif(provino_id, y, occluded, window_size=50, std_threshold=0.08, min_plateau_len=50):
    total_frames = len(y)
    occ_fraction = np.sum(occluded) / total_frames if total_frames > 0 else 1.0
    
    breaking_load = BREAKING_LOADS.get(provino_id, 0.0)
    exp_loads = get_expected_loads(breaking_load)
    exp_seq = get_expected_seq(exp_loads)
    
    if occ_fraction > 0.85:
        return {
            "status": "ERRORE TRACCIAMENTO",
            "reason": f"Tracciamento fallito (occlusione al {occ_fraction*100:.1f}%)",
            "compliance": 2.7,
            "plateaus": [],
            "mapped_loads": [],
            "expected_seq": exp_seq,
            "max_detected_load": 0.0,
            "breaking_load": breaking_load,
            "occ_fraction": occ_fraction
        }
        
    plats = detect_plateaus_verif(y, occluded, window_size, std_threshold, min_plateau_len)
    plat_vals = [p[3] for p in plats]
    
    if not plat_vals or (len(plat_vals) == 1 and abs(plat_vals[0]) < 0.3):
        return {
            "status": "ERRORE TRACCIAMENTO",
            "reason": "Nessun plateau stabile rilevato oltre a quello iniziale",
            "compliance": 2.7,
            "plateaus": plats,
            "mapped_loads": [0.0] * len(plat_vals),
            "expected_seq": exp_seq,
            "max_detected_load": 0.0,
            "breaking_load": breaking_load,
            "occ_fraction": occ_fraction
        }
        
    allowed_loads = sorted(list(set(exp_seq)))
    best_C = 2.7
    best_err = float('inf')
    best_mapped = []
    subseq_match = False
    
    # Try with subsequence constraint
    for C in np.linspace(2.0, 3.5, 151):
        mapped = []
        err = 0.0
        for val in plat_vals:
            diffs = [abs(val - C * L) for L in allowed_loads]
            idx = np.argmin(diffs)
            mapped.append(allowed_loads[idx])
            err += diffs[idx]**2
        if is_subsequence(mapped, exp_seq):
            if err < best_err:
                best_err = err
                best_C = C
                best_mapped = mapped
                subseq_match = True
                
    # Fallback if no valid subsequence match
    if not subseq_match:
        for C in np.linspace(2.0, 3.5, 151):
            mapped = []
            err = 0.0
            for val in plat_vals:
                diffs = [abs(val - C * L) for L in allowed_loads]
                idx = np.argmin(diffs)
                mapped.append(allowed_loads[idx])
                err += diffs[idx]**2
            if err < best_err:
                best_err = err
                best_C = C
                best_mapped = mapped
                
    max_detected_load = max(best_mapped) if best_mapped else 0.0
    
    # Determine status
    if not subseq_match:
        status = "ANOMALO"
        reason = "I carichi rilevati non seguono la sequenza temporale prevista"
    else:
        non_zero_mapped = [l for l in best_mapped if l > 0.0]
        if not non_zero_mapped:
            status = "ERRORE TRACCIAMENTO"
            reason = "Nessun carico stabile rilevato"
        else:
            last_active_load = non_zero_mapped[-1]
            
            # Un provino è CORRETTO se raggiunge il carico nominale previsto, altrimenti è PARZIALE (rottura anticipata)
            if max_detected_load >= breaking_load:
                status = "CORRETTO"
                reason = f"Coincide con agent.md (raggiunto il carico nominale di {breaking_load} kg)"
            else:
                status = "PARZIALE"
                reason = f"Sequenza coerente, ma rottura anticipata a {max_detected_load} kg (atteso {breaking_load} kg)"
                
    return {
        "status": status,
        "reason": reason,
        "compliance": best_C,
        "plateaus": plats,
        "mapped_loads": best_mapped,
        "expected_seq": exp_seq,
        "max_detected_load": max_detected_load,
        "breaking_load": breaking_load,
        "occ_fraction": occ_fraction
    }

# Analizziamo i provini con i parametri iniziali di stabilità
risultati_verifica = {}
for pid, path in disponibili.items():
    if pid in dati_tracking:
        y = dati_tracking[pid]['spostamento']
        occluded = dati_tracking[pid]['occluso']
        risultati_verifica[pid] = analyze_specimen_verif(pid, y, occluded)

print("✅ Inizializzato motore di verifica. Eseguita analisi iniziale su tutti i provini disponibili.")


✅ Inizializzato motore di verifica. Eseguita analisi iniziale su tutti i provini disponibili.


In [10]:
# -------------------------------------------------------------
# 7. Riepilogo Classificazione (agent.md vs Misurato)
# -------------------------------------------------------------
print("="*80)
print(f"{'PROVINO':<10} | {'CARICO NOMINALE':<18} | {'MAX RILEVATO':<15} | {'STATO MAPPATURA':<20} | {'NOTE / DIAGNOSTICA'}")
print("="*80)

corretti = 0
parziali = 0
falliti = 0

for pid in sorted(list(BREAKING_LOADS.keys())):
    if pid not in dati_tracking:
        status = "ERRORE TRACCIAMENTO"
        reason = "File di tracking (.npy) non presente"
        max_detected = 0.0
        falliti += 1
    else:
        y = dati_tracking[pid]['spostamento']
        occluded = dati_tracking[pid]['occluso']
        res = analyze_specimen_verif(pid, y, occluded)
        status = res['status']
        reason = res['reason']
        max_detected = res['max_detected_load']
        
        if status == "CORRETTO":
            corretti += 1
        elif status == "PARZIALE":
            parziali += 1
        else:
            falliti += 1
            
    nominal = BREAKING_LOADS[pid]
    print(f"{pid:<10} | {f'{nominal:.2f} kg':<18} | {f'{max_detected:.2f} kg':<15} | {status:<20} | {reason}")

print("="*80)
print(f"Riepilogo: {corretti} Corretti, {parziali} Parziali, {falliti} Errori/Anomalie su {len(BREAKING_LOADS)} provini.")
print("="*80)

PROVINO    | CARICO NOMINALE    | MAX RILEVATO    | STATO MAPPATURA      | NOTE / DIAGNOSTICA
101        | 2.50 kg            | 0.00 kg         | ERRORE TRACCIAMENTO  | File di tracking (.npy) non presente
102        | 2.50 kg            | 0.00 kg         | ERRORE TRACCIAMENTO  | File di tracking (.npy) non presente
301        | 1.75 kg            | 0.00 kg         | ERRORE TRACCIAMENTO  | File di tracking (.npy) non presente
302        | 2.00 kg            | 0.00 kg         | ERRORE TRACCIAMENTO  | File di tracking (.npy) non presente
303        | 1.75 kg            | 0.00 kg         | ERRORE TRACCIAMENTO  | File di tracking (.npy) non presente
304        | 2.00 kg            | 0.00 kg         | ERRORE TRACCIAMENTO  | File di tracking (.npy) non presente
601        | 2.00 kg            | 0.00 kg         | ERRORE TRACCIAMENTO  | File di tracking (.npy) non presente
602        | 2.00 kg            | 0.00 kg         | ERRORE TRACCIAMENTO  | File di tracking (.npy) non presente
901       

In [11]:
# -------------------------------------------------------------
# 8. Controllo Grafico dei Pesi Applicati
# -------------------------------------------------------------
import matplotlib.pyplot as plt

def mostra_verifica_provino(provino_id):
    if provino_id not in dati_tracking:
        print(f"⚠️ Dati non disponibili per il provino {provino_id}")
        return
        
    y = dati_tracking[provino_id]['spostamento']
    occluded = dati_tracking[provino_id]['occluso']
    
    res = analyze_specimen_verif(provino_id, y, occluded)
    C = res['compliance']
    plats = res['plateaus']
    mapped = res['mapped_loads']
    
    fig, ax = plt.subplots(figsize=(12, 5))
    
    # Disegna lo spostamento attivo ed occluso
    ax.plot(y, color='#1f77b4', linewidth=2, label='Tracking Attivo')
    y_occ = np.where(occluded, y, np.nan)
    ax.plot(y_occ, color='#ff7f0e', linewidth=2.5, label='Occlusione')
    
    # Linee di riferimento per carichi teorici calibrati
    max_level = res['breaking_load'] + 0.5
    for L in [0.5, 1.25, 1.75, 2.0, 2.5, 3.25, 3.75]:
        if L <= max_level:
            disp_level = C * L
            ax.axhline(disp_level, color='#7f8c8d', linestyle='--', alpha=0.5)
            ax.text(len(y) * 1.002, disp_level, f"{L:.2f} kg", color='#7f8c8d', fontsize=9, va='center')
            
    # Evidenzia i plateau rilevati ed etichetta i pesi
    for idx, (start, end, _, mean_val) in enumerate(plats):
        mapped_L = mapped[idx]
        weight_str = WEIGHT_COMPOSITIONS.get(mapped_L, f"{mapped_L} kg")
        
        ax.axvspan(start, end, color='#2ecc71', alpha=0.1)
        ax.hlines(mean_val, start, end, colors='#2ecc71', linewidth=3)
        
        # Calcolo dell'offset verticale per la label
        y_offset = 0.2 if mean_val >= 0 else -0.4
        ax.text((start + end) / 2, mean_val + y_offset, f"{mapped_L} kg\n({weight_str})", 
                color='#2c3e50', fontsize=8.5, fontweight='bold', ha='center', va='bottom')

    # 1. Evidenziazione post-rottura (quando l'ultimo carico attivo termina)
    attivi = [i for i, L in enumerate(mapped) if L > 0.0]
    if attivi:
        last_active_idx = attivi[-1]
        _, end_frame, _, _ = plats[last_active_idx]
        if end_frame < len(y):
            ax.axvline(end_frame, color='#e74c3c', linestyle='--', linewidth=2, label='Instante di Rottura')
            ax.axvspan(end_frame, len(y), color='#e74c3c', alpha=0.1)
            ax.text(end_frame + (len(y) - end_frame)/2, 0.8, 'Zona Post-Rottura\n(Dati non validi)',
                    color='#e74c3c', fontsize=9.5, ha='center', va='center', fontstyle='italic',
                    transform=ax.get_xaxis_transform())

    # 2. Box informativo in basso a destra con il carico nominale da agent.md
    nominal_load = BREAKING_LOADS.get(provino_id, 0.0)
    ax.text(0.98, 0.05, f"Carico Rottura Teorico (agent.md): {nominal_load:.2f} kg",
            transform=ax.transAxes, ha='right', va='bottom', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.3", fc="#f8f9fa", ec="#e74c3c", alpha=0.9))

    ax.set_title(f"Verifica Carichi - Provino {provino_id} (Stato: {res['status']})", fontsize=12, fontweight='bold')
    ax.set_xlabel("Frame")
    ax.set_ylabel("Spostamento (mm)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left')
    ax.set_xlim(0, len(y) * 1.05)
    plt.show()

# Dropdown interattivo
provini_presenti = sorted(list(risultati_verifica.keys()))
if provini_presenti:
    widgets.interact(mostra_verifica_provino, provino_id=widgets.Dropdown(options=provini_presenti, value=provini_presenti[0], description="Provino:"))
else:
    print("Nessun provino tracciato.")

Nessun provino tracciato.


## 7. Estrazione Relazione Carico-Allungamento e Modulo di Young

In questa sezione raccogliamo ed elaboriamo i dati di deflessione verticale registrati per ciascun livello di carico (plateau stabili) per tutti i provini.
Per ogni provino viene eseguita una regressione lineare (Legge di Hooke: $\Delta L = C \cdot L + d$):
* **Cedevolezza ($C$, mm/kg)**: Pendenza della retta di regressione.
* **Rigidezza ($k$, N/mm)**: Calcolata come $k = g / C$, con $g = 9.80665$ m/s$^2$.
* **Modulo di Young ($E$)**:
  * **Modello a Trazione Uniassiale**: $E_{\text{trazione}} = \frac{g \cdot L_0}{A \cdot C}$ (MPa), con $A = w \cdot t$.
  * **Modello a Flessione (Cantilever Bending)**: $E_{\text{flessione}} = \frac{g \cdot L_b^3}{3 \cdot I \cdot C \cdot 1000}$ (GPa), con $I = \frac{w \cdot t^3}{12}$.

In [12]:
import numpy as np
import pandas as pd
from scipy.stats import linregress

# ==========================================
# PARAMETRI GEOMETRICI E FISICI DEI PROVINI
# ==========================================
# Puoi modificare questi parametri per riflettere le dimensioni reali del tuo setup
L_0 = 50.0          # Lunghezza utile iniziale per modello a trazione (mm)
L_b = 90.0          # Lunghezza a sbalzo libera per modello a flessione (mm)
width_mm = 10.0     # Larghezza nominale della sezione (mm)
thickness_mm = 4.0  # Spessore nominale della sezione (mm)
g = 9.80665         # Accelerazione di gravità (m/s^2)

# Calcolo geometrico derivato
A_mm2 = width_mm * thickness_mm
I_mm4 = (width_mm * (thickness_mm ** 3)) / 12.0

# Carichi teorici e provini disponibili
all_loads = [0.5, 1.25, 1.75, 2.0, 2.5, 3.25, 3.75]

specimens_results = {}

for pid in sorted(list(dati_tracking.keys())):
    y = dati_tracking[pid]['spostamento']
    occluded = dati_tracking[pid]['occluso']
    
    # Eseguiamo la ricerca dei plateau come configurata nel notebook
    res = analyze_specimen_verif(pid, y, occluded)
    C_mapped = res['compliance']
    plats = res['plateaus']
    mapped = res['mapped_loads']
    
    # Raccogliamo i punti sperimentali
    x_pts = []
    y_pts = []
    load_to_disps = {L: [] for L in [0.0] + all_loads}
    for idx, (_, _, _, mean_val) in enumerate(plats):
        if idx < len(mapped):
            m_load = mapped[idx]
            x_pts.append(m_load)
            y_pts.append(mean_val)
            load_to_disps[m_load].append(mean_val)
            
    if len(x_pts) >= 2:
        slope, intercept, r_val, p_val, std_err = linregress(x_pts, y_pts)
        r2 = r_val ** 2
    else:
        slope, intercept, r2 = 0.0, 0.0, 0.0
        
    stiffness_n_mm = g / slope if slope != 0 else 0.0
    E_trazione_mpa = stiffness_n_mm * L_0 / A_mm2
    E_flessione_gpa = (stiffness_n_mm * (L_b ** 3) / (3.0 * I_mm4)) / 1000.0 if I_mm4 != 0 else 0.0
    
    breaking_load = BREAKING_LOADS.get(pid, 0.0)
    sigma_rupture_mpa = (breaking_load * g) / A_mm2
    
    avg_disps = {}
    for L in all_loads:
        vals = load_to_disps[L]
        avg_disps[L] = np.mean(vals) if vals else np.nan
        
    specimens_results[pid] = {
        'x_pts': x_pts,
        'y_pts': y_pts,
        'slope': slope,
        'intercept': intercept,
        'r2': r2,
        'stiffness': stiffness_n_mm,
        'E_trazione': E_trazione_mpa,
        'E_flessione': E_flessione_gpa,
        'breaking_load': breaking_load,
        'sigma_rupture': sigma_rupture_mpa,
        'avg_disp': avg_disps
    }

# 1. Creazione Tabella degli Allungamenti
rows_elong = []
for pid, r in specimens_results.items():
    row = {'Provino': pid}
    for L in all_loads:
        val = r['avg_disp'][L]
        row[f"{L:.2f} kg"] = f"{val:.3f} mm" if not np.isnan(val) else "-"
    rows_elong.append(row)

df_elong = pd.DataFrame(rows_elong)

# 2. Creazione Tabella dei Parametri Meccanici
rows_mech = []
for pid, r in specimens_results.items():
    row = {
        'Provino': pid,
        'Carico Rottura (kg)': r['breaking_load'],
        'Tensione Rottura (MPa)': r['sigma_rupture'],
        'Cedevolezza C (mm/kg)': r['slope'],
        'Rigidezza k (N/mm)': r['stiffness'],
        'E_trazione (MPa)': r['E_trazione'],
        'E_flessione (GPa)': r['E_flessione'],
        'R² Lineare': r['r2']
    }
    rows_mech.append(row)

df_mech = pd.DataFrame(rows_mech)

print("="*60)
print("TABELLA 1: ALLUNGAMENTO RILEVATO PER LIVELLO DI CARICO")
print("="*60)
display(df_elong)
print("\n" + "="*60)
print("TABELLA 2: PARAMETRI MECCANICI ED ELASTICI CALCOLATI")
print("="*60)
display(df_mech.style.format({
    'Carico Rottura (kg)': '{:.2f}',
    'Tensione Rottura (MPa)': '{:.3f}',
    'Cedevolezza C (mm/kg)': '{:.4f}',
    'Rigidezza k (N/mm)': '{:.2f}',
    'E_trazione (MPa)': '{:.1f}',
    'E_flessione (GPa)': '{:.2f}',
    'R² Lineare': '{:.4f}'
}))

TABELLA 1: ALLUNGAMENTO RILEVATO PER LIVELLO DI CARICO


""



TABELLA 2: PARAMETRI MECCANICI ED ELASTICI CALCOLATI


In [13]:
import matplotlib.pyplot as plt

def mostra_regressione_hooke(provino_selezionato):
    plt.close('all')
    
    if provino_selezionato == "Confronto Tutti":
        # Grafico comparativo
        fig, ax = plt.subplots(figsize=(12, 7))
        colors = plt.cm.tab10(np.linspace(0, 1, len(specimens_results)))
        
        for idx, (pid, r) in enumerate(specimens_results.items()):
            x = np.array(r['x_pts'])
            y = np.array(r['y_pts'])
            
            # Scatter dei punti reali
            ax.scatter(x, y, color=colors[idx], alpha=0.7, edgecolors='k')
            
            # Retta di regressione
            x_fit = np.linspace(0, max(x) if len(x)>0 else 2.5, 100)
            y_fit = r['slope'] * x_fit + r['intercept']
            ax.plot(x_fit, y_fit, color=colors[idx], linewidth=1.5,
                    label=f"Provino {pid} (k={r['stiffness']:.2f} N/mm, R²={r['r2']:.3f})")
            
        ax.set_title("Confronto Curve di Hooke - Tutti i Provini", fontsize=13, fontweight='bold')
        ax.set_xlabel("Carico (kg)", fontsize=11)
        ax.set_ylabel("Allungamento / Deflessione (mm)", fontsize=11)
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9.5)
        plt.tight_layout()
        plt.show()
        
    else:
        # Grafico singolo provino con dettagli
        r = specimens_results[provino_selezionato]
        x = np.array(r['x_pts'])
        y = np.array(r['y_pts'])
        
        fig, ax = plt.subplots(figsize=(10, 5.5))
        
        # Punti
        ax.scatter(x, y, color='#1f77b4', s=70, zorder=3, label='Dati Rilevati (Plateau)')
        
        # Linea fit
        x_fit = np.linspace(0, max(x) * 1.1 if len(x)>0 else 2.5, 100)
        y_fit = r['slope'] * x_fit + r['intercept']
        ax.plot(x_fit, y_fit, color='#d62728', linewidth=2.5, zorder=2,
                label=f'Fitted Line: y = {r["slope"]:.4f}*x + {r["intercept"]:.4f}')
        
        # Titoli e dettagli
        ax.set_title(f"Legge di Hooke & Regressione - Provino {provino_selezionato}", fontsize=13, fontweight='bold')
        ax.set_xlabel("Carico (kg)", fontsize=11)
        ax.set_ylabel("Allungamento / Deflessione (mm)", fontsize=11)
        ax.set_xlim(0, max(x_fit) if len(x_fit)>0 else 3.0)
        ax.set_ylim(0, max(y_fit) * 1.1 if len(y_fit)>0 else 8.0)
        
        # Box testo informativo
        box_text = (
            f"Cedevolezza C: {r['slope']:.4f} mm/kg\n"
            f"Rigidezza k: {r['stiffness']:.2f} N/mm\n"
            f"E (Trazione): {r['E_trazione']:.1f} MPa\n"
            f"E (Flessione): {r['E_flessione']:.2f} GPa\n"
            f"Coefficiente R²: {r['r2']:.5f}"
        )
        ax.text(0.05, 0.95, box_text, transform=ax.transAxes, fontsize=10,
                verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', facecolor='#f8f9fa', edgecolor='#ccc', alpha=0.9))
        
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(loc='lower right', fontsize=10)
        plt.tight_layout()
        plt.show()

# Creazione dropdown interattivo
opzioni_plot = ["Confronto Tutti"] + sorted(list(specimens_results.keys()))
widgets.interact(mostra_regressione_hooke, 
                 provino_selezionato=widgets.Dropdown(options=opzioni_plot, value="Confronto Tutti", description="Visualizza:"))

interactive(children=(Dropdown(description='Visualizza:', options=('Confronto Tutti',), value='Confronto Tutti…

<function __main__.mostra_regressione_hooke(provino_selezionato)>

### 8. Visualizzazione Interattiva dei Parametri Meccanici (Ordinamento e Filtro)

In questa cella finale è possibile filtrare i provini in base al tempo di stoppage (difetto di stampa artificiale) e ordinarli secondo vari parametri fisici rilevati. La tabella applica una mappa di colori gradiente (plasma) sulla rigidezza e sul modulo di Young a flessione per identificare visivamente i provini strutturalmente più rigidi (colori caldi) rispetto a quelli più cedevoli (colori freddi).

In [14]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Prepariamo il dataframe completo di informazioni sul difetto (tempo di stoppage)
df_visual = df_mech.copy()
df_visual['Stoppage (s)'] = df_visual['Provino'].apply(lambda x: 0 if x.startswith("99") else int(x[:2]))

# Riordina le colonne per posizionare lo Stoppage subito dopo il codice del Provino
cols = ['Provino', 'Stoppage (s)', 'Carico Rottura (kg)', 'Tensione Rottura (MPa)', 
        'Cedevolezza C (mm/kg)', 'Rigidezza k (N/mm)', 'E_trazione (MPa)', 'E_flessione (GPa)', 'R² Lineare']
df_visual = df_visual[cols]

def aggiorna_tabella(ordina_per, filtra_per):
    clear_output(wait=True)
    df_filtered = df_visual.copy()
    
    # Applicazione del filtro per tempo di stoppage
    if filtra_per != "Tutti":
        val_stoppage = int(filtra_per.replace("s", ""))
        df_filtered = df_filtered[df_filtered['Stoppage (s)'] == val_stoppage]
        
    # Applicazione dell'ordinamento selezionato
    if ordina_per == "Rigidezza (Decrescente)":
        df_filtered = df_filtered.sort_values(by='Rigidezza k (N/mm)', ascending=False)
    elif ordina_per == "Rigidezza (Crescente)":
        df_filtered = df_filtered.sort_values(by='Rigidezza k (N/mm)', ascending=True)
    elif ordina_per == "Stoppage (Crescente)":
        df_filtered = df_filtered.sort_values(by='Stoppage (s)', ascending=True)
    elif ordina_per == "R² Lineare (Decrescente)":
        df_filtered = df_filtered.sort_values(by='R² Lineare', ascending=False)
        
    # Definizione dello stile per la visualizzazione della tabella
    style_dict = df_filtered.style.format({
        'Carico Rottura (kg)': '{:.2f}',
        'Tensione Rottura (MPa)': '{:.3f}',
        'Cedevolezza C (mm/kg)': '{:.4f}',
        'Rigidezza k (N/mm)': '{:.2f}',
        'E_trazione (MPa)': '{:.1f}',
        'E_flessione (GPa)': '{:.2f}',
        'R² Lineare': '{:.4f}'
    }).background_gradient(
        cmap='plasma_r',  # Mappa di colori plasma invertita per evidenziare i provini migliori
        subset=['Rigidezza k (N/mm)', 'E_flessione (GPa)']
    ).set_properties(**{
        'text-align': 'center',
        'font-family': 'DejaVu Sans, Arial, Helvetica, sans-serif',
        'border': '1px solid #ccc'
    })
    
    print(f"🔍 Risultati filtrati per: {filtra_per} | Ordinati per: {ordina_per}\n")
    display(style_dict)

# Creazione dei menu a tendina interattivi
dropdown_ordina = widgets.Dropdown(
    options=[
        "Nessuno (Default)", 
        "Rigidezza (Decrescente)", 
        "Rigidezza (Crescente)", 
        "Stoppage (Crescente)", 
        "R² Lineare (Decrescente)"
    ],
    value="Nessuno (Default)",
    description="Ordina per:"
)

dropdown_filtra = widgets.Dropdown(
    options=["Tutti", "0s", "10s", "30s", "60s", "90s"],
    value="Tutti",
    description="Stoppage:"
)

# Layout e visualizzazione dei controlli
ui = widgets.HBox([dropdown_ordina, dropdown_filtra])
out = widgets.interactive_output(aggiorna_tabella, {
    'ordina_per': dropdown_ordina,
    'filtra_per': dropdown_filtra
})

display(ui, out)

KeyError: 'Provino'

### 9. Export Dati Finali (Proprietà Meccaniche)

Salva su disco i risultati finali dell'analisi meccanica:
- **Proprietà meccaniche** (`mechanical_properties.csv`): cedevolezza, rigidezza, modulo di Young (trazione e flessione), tensione di rottura, R² per ogni provino.
- **Tabella allungamenti** (`elongation_table.csv`): allungamento rilevato per ogni livello di carico per ogni provino.
- **Verifica carichi** (`load_verification.csv`): stato della mappatura (CORRETTO/PARZIALE/ERRORE), max carico rilevato, compliance.

In [ ]:
import os
import pandas as pd

EXPORT_DIR = os.path.join("dati", "dataset_finale", "meccanica")
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Salva le proprietà meccaniche
# Aggiungiamo la colonna Stoppage (s) per il terzo notebook
df_mech_export = df_mech.copy()
df_mech_export['Stoppage (s)'] = df_mech_export['Provino'].apply(
    lambda x: 0 if x.startswith("99") else int(x[:2])
)
df_mech_export.to_csv(os.path.join(EXPORT_DIR, "mechanical_properties.csv"), index=False)
print(f"Salvato mechanical_properties.csv ({len(df_mech_export)} righe)")

# 2. Salva la tabella degli allungamenti
df_elong.to_csv(os.path.join(EXPORT_DIR, "elongation_table.csv"), index=False)
print(f"Salvato elongation_table.csv ({len(df_elong)} righe)")

# 3. Salva i risultati della verifica carichi
verif_rows = []
for pid in sorted(list(BREAKING_LOADS.keys())):
    if pid in dati_tracking:
        y = dati_tracking[pid]['spostamento']
        occluded = dati_tracking[pid]['occluso']
        res = analyze_specimen_verif(pid, y, occluded)
        verif_rows.append({
            "provino_id": pid,
            "carico_nominale_kg": res['breaking_load'],
            "max_carico_rilevato_kg": res['max_detected_load'],
            "stato": res['status'],
            "compliance_mm_kg": res['compliance'],
            "n_plateau": len(res['plateaus']),
            "occ_fraction": res['occ_fraction'],
            "nota": res['reason']
        })
    else:
        verif_rows.append({
            "provino_id": pid,
            "carico_nominale_kg": BREAKING_LOADS[pid],
            "max_carico_rilevato_kg": 0.0,
            "stato": "MANCANTE",
            "compliance_mm_kg": 0.0,
            "n_plateau": 0,
            "occ_fraction": 1.0,
            "nota": "Dati di tracking non disponibili"
        })

df_verif = pd.DataFrame(verif_rows)
df_verif.to_csv(os.path.join(EXPORT_DIR, "load_verification.csv"), index=False)
print(f"Salvato load_verification.csv ({len(df_verif)} righe)")

print(f"\n✅ Export meccanica completato in: {EXPORT_DIR}")


Salvato mechanical_properties.csv (11 righe)
Salvato elongation_table.csv (11 righe)
Salvato load_verification.csv (12 righe)

✅ Export meccanica completato in: dati\dataset_finale\meccanica
